<a href="https://colab.research.google.com/github/kavanaknaik-sketch/AI-LAB/blob/main/manhattan%20distance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import heapq

class PuzzleNode:
    def __init__(self, state, g_cost, goal_state, parent=None):
        self.state = state        # Current board layout (tuple)
        self.g_cost = g_cost      # Depth of the node g(n)
        self.goal_state = goal_state
        self.parent = parent      # Link to track path steps
        self.h_cost = self._calculate_manhattan()
        self.f_cost = self.g_cost + self.h_cost  # f(n) = g(n) + h(n)

    def _calculate_manhattan(self):
        # Sum of the horizontal and vertical distances of each tile from its target position
        distance = 0
        for i in range(9):
            tile = self.state[i]
            if tile != 0:  # Ignore the blank space '0'
                # Current 2D coordinates (row, col)
                curr_row, curr_col = i // 3, i % 3

                # Goal 2D coordinates (row, col)
                goal_index = self.goal_state.index(tile)
                goal_row, goal_col = goal_index // 3, goal_index % 3

                # Manhattan distance formula: |x1 - x2| + |y1 - y2|
                distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
        return distance

    def __lt__(self, other):
        return self.f_cost < other.f_cost

def get_neighbors(state):
    """Generates valid adjacent board arrangements by moving the blank spot (0)."""
    neighbors = []
    blank_idx = state.index(0)
    row, col = blank_idx // 3, blank_idx % 3

    # Valid moves: Up, Down, Left, Right
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in moves:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col
            new_state = list(state)
            new_state[blank_idx], new_state[new_idx] = new_state[new_idx], new_state[blank_idx]
            neighbors.append(tuple(new_state))

    return neighbors

def is_solvable(start, goal):
    """Checks if a puzzle can transition from start to goal layout via inversion parity."""
    def count_inversions(state):
        # Exclude blank tile (0) for counting inversions
        flat_list = [tile for tile in state if tile != 0]
        inversions = 0
        for i in range(len(flat_list)):
            for j in range(i + 1, len(flat_list)):
                if flat_list[i] > flat_list[j]:
                    inversions += 1
        return inversions

    return (count_inversions(start) % 2) == (count_inversions(goal) % 2)

def solve_a_star(start_state, goal_state):
    """Executes the search using Manhattan Distance algorithm framework."""
    if not is_solvable(start_state, goal_state):
        return None, 0

    start_node = PuzzleNode(start_state, 0, goal_state)
    open_list = [start_node]
    closed_set = set()
    nodes_expanded = 0

    while open_list:
        current_node = heapq.heappop(open_list)
        nodes_expanded += 1

        if current_node.state == goal_state:
            # Reconstruct the path backwards
            path = []
            while current_node:
                path.append(current_node.state)
                current_node = current_node.parent
            return path[::-1], nodes_expanded

        closed_set.add(current_node.state)

        for neighbor_state in get_neighbors(current_node.state):
            if neighbor_state in closed_set:
                continue

            neighbor_node = PuzzleNode(neighbor_state, current_node.g_cost + 1, goal_state, current_node)
            heapq.heappush(open_list, neighbor_node)

    return None, nodes_expanded

def print_board(state):
    """Outputs the 9-element tuple sequentially into clean 3x3 tiles."""
    for i in range(0, 9, 3):
        print(f" {state[i]}  {state[i+1]}  {state[i+2]}")
    print()

def get_user_input(prompt_text):
    """Safely captures inputs and converts space-separated strings to integer tuples."""
    print(prompt_text)
    print("Example layout format (use 0 for blank): 1 2 3 4 5 6 7 8 0")
    print("1WN2CS134 Kavana K Naik")
    while True:
        try:
            user_in = input("Enter 9 numbers separated by spaces: ").strip().split()
            nums = [int(x) for x in user_in]
            if len(nums) == 9 and sorted(nums) == list(range(9)):
                return tuple(nums)
            print("❌ Invalid input! Ensure you enter exactly numbers 0 through 8 with no duplicates.")
        except ValueError:
            print("❌ Invalid characters! Numbers only.")

# --- Main Driver Script Block ---
if __name__ == "__main__":
    print("=== 8-Puzzle Solver: Manhattan Distance Heuristic ===\n")

    # 1. Capture customized target coordinates
    initial_state = get_user_input("--- Setup Initial State ---")
    goal_state = get_user_input("\n--- Setup Target Goal State ---")

    print("\nProcessing path solutions...")
    solution_path, nodes_visited = solve_a_star(initial_state, goal_state)

    # 2. Map and display iterative solution movements
    if solution_path:
        print(f"\n✅ Solution found in {len(solution_path) - 1} moves!")
        print(f"📊 Nodes Expanded (States Evaluated): {nodes_visited}\n")
        for step, state in enumerate(solution_path):
            if step == 0:
                print(f"[Start: Initial Board]")
            elif step == len(solution_path) - 1:
                print(f"[Step {step}: Goal Reached!]")
            else:
                print(f"[Step {step}]")
            print_board(state)
    else:
        print("\n❌ This layout combination is unsolvable. It is mathematically impossible to reach the goal from your starting point.")


=== 8-Puzzle Solver: Manhattan Distance Heuristic ===

--- Setup Initial State ---
Example layout format (use 0 for blank): 1 2 3 4 5 6 7 8 0
1WN2CS134 Kavana K Naik
Enter 9 numbers separated by spaces: 1 2 3 4 5 6 7 8 0

--- Setup Target Goal State ---
Example layout format (use 0 for blank): 1 2 3 4 5 6 7 8 0
1WN2CS134 Kavana K Naik
Enter 9 numbers separated by spaces: 1 2 3 4 5 6 0 7 8

Processing path solutions...

✅ Solution found in 2 moves!
📊 Nodes Expanded (States Evaluated): 3

[Start: Initial Board]
 1  2  3
 4  5  6
 7  8  0

[Step 1]
 1  2  3
 4  5  6
 7  0  8

[Step 2: Goal Reached!]
 1  2  3
 4  5  6
 0  7  8

